### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [36]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model



ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11e00ce90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11e00d810>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [37]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    name: str = Field(description="The name of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The rating of the movie")

In [38]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11e00ce90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11e00d810>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'name': {'description': 'The name of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie', 'type': 'number'}}, 'required': ['name', 'year', 'director', 'rating'], 

In [39]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. It\'s a 2010 movie directed by Christopher Nolan, right? The main actor is Leonardo DiCaprio. The title "Inception" makes me think it\'s about planting ideas or something related to the mind. I remember it\'s a sci-fi action film. There\'s a lot of dream sequences, maybe using some kind of technology to enter people\'s dreams.\n\nThe plot probably involves a thief who steals information by entering people\'s dreams, and then maybe he has to do the opposite, which is planting an idea. That\'s the inception part. The main character\'s name is Dom Cobb, played by DiCaprio. He has a family, but I think his wife is a problem for him because of some guilt. There\'s a scene where he keeps seeing her, which might be a manifestation of his subconscious.\n\nThe team of characters includes a lot of actors like Joseph Gordon-Levitt, Ellen Page, Tom Hardy, and ot

In [40]:
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(name='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside structure

In [41]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    name: str = Field(description="The name of the movie")
    director: str = Field(description="The director of the movie")
    release_year: int = Field(description="The release year of the movie")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="The rating of the movie")

model_with_structure=model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking for details about the movie Inception. I need to use the Movie function provided. The required parameters are name, director, release_year, genre, and rating.\n\nFirst, I know the name is "Inception". The director is Christopher Nolan. The release year was 2010. The genre is science fiction or maybe action. The rating is probably high, like 8.8 on IMDb. I should make sure all required fields are included. Let me double-check each parameter. Yeah, that\'s all covered. Let me format the JSON correctly.\n', 'tool_calls': [{'id': '5dfv1ebxf', 'function': {'arguments': '{"director":"Christopher Nolan","genre":"Science Fiction","name":"Inception","rating":8.8,"release_year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 180, 'prompt_tokens': 246, 'total_tokens': 426, 'completion_time': 0.295765591, 'completion_tokens_details': {

### Nested Structure

In [42]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(description="The name of the actor")
    age: int = Field(description="The age of the actor")
    gender: str = Field(description="The gender of the actor")
    role: str = Field(description="The role of the actor")

class MovieDetails(BaseModel):
   title: str = Field(description="The title of the movie")
   year: int = Field(description="The year of the movie")
   genre: list[str] = Field(description="The genre of the movie")
   rating: float = Field(description="The rating of the movie")
   actors: list[Actor] = Field(description="The actors of the movie")
   budget: float = Field(description="Budget in millions USD")
   box_office: float = Field(description="Box office in millions USD")
   director: str = Field(description="The director of the movie")
   writer: str = Field(description="The writer of the movie")
   production_company: str = Field(description="The production company of the movie")
   


model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, genre=['Action', 'Science Fiction'], rating=8.8, actors=[Actor(name='Leonardo DiCaprio', age=49, gender='Male', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', age=46, gender='Male', role='Arthur'), Actor(name='Ellen Page', age=38, gender='Female', role='Ariadne'), Actor(name='Tom Hardy', age=44, gender='Male', role='Balthazar')], budget=160.0, box_office=886.7, director='Christopher Nolan', writer='Christopher Nolan', production_company='Warner Bros.')

### TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [43]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [44]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}

In [45]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [46]:
import os
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [47]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='e0ed83ed-a38f-4c5e-82b6-7c6d1c7d78e6'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 676, 'prompt_tokens': 204, 'total_tokens': 880, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CsghrMKkjrFgLdCcSyQ5Y1Mb1kF9K', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b7257-bc3b-7d52-b2de-ed91dc51ce26-0', usage_metadata={'input_tokens': 204, 'output_tokens': 676, 'total_tokens': 880, 'i

In [48]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [49]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [50]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')